# Model Heuristics and Rankings 

this notebook describes features from our merged_df considered and how they are prioritised in our ranking system. 

Simple weighted heuristics based system rather than model.... happy to accept other ideas and hear challenges on this. 

Assumptions: 
- BNPL company gets payed a percentage of each transaction and late fees from consumers (lots of revenue, lots of customers) 
- Main goal for the company is profit and growth 

Considerations: 
- fraud treated as a 'penalty' not an outright exclusion 
- diverse categories considered to maintain robust merchant profile 
- data from 2021 - 2022 

## Profit target 
Expected profit can be defined as revenue * take rate * (1 - fraud_probability)

In [ ]:
from pyspark.sql import SparkSession, functions as F

def get_spark_session(app_name: str = "SparkSession") -> SparkSession:
    """
    Create a SparkSession with the given app name.

    """
    return SparkSession.builder.appName(app_name).getOrCreate()

spark = get_spark_session()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# get profit target from merged_df and group by category. 

#load merged_df from parquet file
merged_df = spark.read.parquet("data/curated/merged_transactions_sa2.parquet")

#add column for profit value based on formula revenue * take rate * (1-fraud probability)
merged_df = merged_df.withColumn("profit", F.col("revenue") * F.col("take_rate") * (1 - F.col("fraud_probability")))

maybe growth score?? 
(2022 - 2021 revenue) / 2021 revenue

other: customer growth, recent growth (3-6 months)

In [ ]:
# calculate growth score as per the above formula for each merchant and group by category.

## Customer Profile - Late Fee Potential 

other potential ideas: unique customer count, rate of repeat customers, concentration of spending 
other: how many unique postcodes served, mix of incomes 

In [ ]:
# determine high, medium, low and very low income postcodes based on the distribution of income in merged_df.
# high proportion of low- or middle-income customers may indicate higher late fee potential 

#check the distribution of income in merged_df
merged_df.select("Median_tot_fam_inc_weekly").describe().show(vertical=True)

#split into quartiles and assign labels
merged_df = merged_df.withColumn("income_category", F.when(F.col("Median_tot_fam_inc_weekly") < F.percentile_approx("Median_tot_fam_inc_weekly", 0.25), "very_low")
                            .when(F.col("Median_tot_fam_inc_weekly") < F.percentile_approx("Median_tot_fam_inc_weekly", 0.5), "low")
                            .when(F.col("Median_tot_fam_inc_weekly") < F.percentile_approx("Median_tot_fam_inc_weekly", 0.75), "medium")
                            .otherwise("high"))

# calculate proportion of customers in each income category for each merchant and group by category.
income_proportions = merged_df.groupBy("merchant_id", "income_category").count()


## Merchant Stability 
how stable is merchant revenue each month 

In [ ]:
# determine monthly income per merchant and find variance across 2021-2022

## Fraud trend 

is fraud random? or does it always concentrate? is it increasing/decreasing

## Performance according to overall market 

how does a merchant perform related to interest rates rising and falling (stay strong, follow, fall off with high interest rates) and does this map to is_essential or not 

# Combine into a final Score 

split to be decided.... 

overall rank approx: 
- profitability
- customer profile
- stability and performance in terms of market?? 


## Diversity 

pick top 10 per category 

## Final Conditions 

- minimum activity 
- reviews 
- human explanation (transparency) 